# Revision experiments (answers to the review)

**What this runs.** Set `PART` in cell 4:
* `"flhelps"` (~13 GPU-h): *how FL helps*. Courts with 10/25/50% of their data (local vs federated vs centralized), and each court held out of training (unseen court).
* `"baselines"` (~30 GPU-h): *comparison with FedLegal-style algorithms*. FedProx, SCAFFOLD, FedAdam, FedPer and Ditto on the same TSR model, 3 seeds, plus our method and local training with per-case predictions saved for significance tests.
* `"controls"` (~11 GPU-h): single-view 512-token model, so the TSR gain is not just from extra tokens; local/centralized TSR seeds 1-2.

With 2 GPUs the job list is split over both, so wall time is about half the GPU-hours. A session stops launching runs after `BUDGET_HOURS`. **To continue**, attach this notebook's own output as an input and run it again: finished runs are skipped and interrupted ones resume.

**Before running**
1. **Add Input** → your dataset with `judgments_labelled.csv`, and a dataset made from the revised repo zip (Kaggle unzips it).
2. Settings → **Accelerator: GPU T4 x2**, **Internet: On** (to download InLegalBERT).
3. **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed. Results are in the version's **Output** tab (`runs/`, `results/`).


In [ ]:
!pip install -q peft
!pip uninstall -y -q torchao
import os, glob, shutil, subprocess, sys, time, torch
# newest copy of each file (one with the revision features), wherever it was attached
MARK = {"fl_judicial.py": "holdout_court", "run_queue.py": '"revision"', "summarize_results.py": "sel_val_loss",
        "heterogeneity_stats.py": "equivalent_dirichlet_alpha", "significance.py": "Delta_MTL",
        "revision_tables.py": "control512"}
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    good = [h for h in hits if MARK[name] in open(h).read()]
    return (good or [None])[0]
csvs = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
cands = [p for p in csvs if "judgments_labelled" in os.path.basename(p)]
assert cands, f"judgments_labelled*.csv not found under /kaggle/input. CSVs found: {csvs}"
DATA = cands[0]
os.chdir("/kaggle/working"); os.makedirs("scripts", exist_ok=True)
for f in MARK:
    p = find(f); assert p, f"{f} (revised version) not found under /kaggle/input: attach the revised repo"
    shutil.copy(p, f if f in ("fl_judicial.py", "run_queue.py", "summarize_results.py") else f"scripts/{f}")
N_GPU = torch.cuda.device_count()
print("data:", DATA, "| GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "no GPU: set Accelerator = GPU T4 x2"
import urllib.request
urllib.request.urlopen("https://huggingface.co", timeout=10); print("internet: ok")


In [ ]:
# Point 1 of the review: heterogeneity statistics on exactly the 12,500 judgments the models use (CPU, ~5 min)
!python scripts/heterogeneity_stats.py --data "$DATA" --out results/heterogeneity_text --perms 10


In [ ]:
# Resume: merge runs/ from earlier sessions of this notebook (attach their output as input). Nothing needed on the first run.
import pandas as pd
os.makedirs("runs", exist_ok=True)
frames = [pd.read_csv("runs/all_results.csv")] if os.path.exists("runs/all_results.csv") else []
for p in glob.glob("/kaggle/input/**/runs/all_results.csv", recursive=True):
    d = os.path.dirname(p)
    for sub in os.listdir(d):
        if os.path.isdir(os.path.join(d, sub)): shutil.copytree(os.path.join(d, sub), os.path.join("runs", sub), dirs_exist_ok=True)
    for f in glob.glob(os.path.join(d, "tok_cache_*.pkl")): shutil.copy(f, "runs/")
    frames.append(pd.read_csv(p)); print("restored", d)
if frames:
    pd.concat(frames, ignore_index=True).drop_duplicates(subset=["run", "method", "court"], keep="last").to_csv("runs/all_results.csv", index=False)
done = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/per_court_test.csv")]
partial = [d for d in os.listdir("runs") if os.path.exists(f"runs/{d}/ckpt.pt") and d not in done]
print(f"finished runs: {len(done)} | will resume: {partial}")


In [ ]:
PART = "flhelps"        # "flhelps" | "baselines" | "controls"
BUDGET_HOURS = 10.5     # stop launching new runs after this (Kaggle limit is 12 h)
def launch(gpu, shard, n):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = [sys.executable, "-u", "run_queue.py", "--data", DATA, "--plan", "revision", "--part", PART,
           "--budget_hours", str(BUDGET_HOURS), "--shard", str(shard), "--n_shards", str(n)]
    return subprocess.Popen(cmd, env=env, stdout=open(f"queue_gpu{gpu}.log", "a"), stderr=subprocess.STDOUT)
n = min(N_GPU, 2); t0 = time.time()
procs = [launch(i, i, n) for i in range(n)]
print(f"launched {n} queue(s) for PART={PART}")


In [ ]:
# Monitor (prints every 10 minutes) and fail loudly if a run crashed
import re
while any(p.poll() is None for p in procs):
    time.sleep(600)
    print(f"\n===== {(time.time()-t0)/3600:.2f} h =====")
    for log in sorted(glob.glob("queue_gpu*.log")):
        lines = [l for l in open(log).read().splitlines() if re.search(r"shard|start|mean:|lambda|early stop|best round|Error|skipped", l)]
        print(log, "\n  " + "\n  ".join(lines[-6:]))
print("queues finished")
errs = [l for l in glob.glob("logs/*.log") + glob.glob("queue_gpu*.log") if "Traceback" in open(l).read()]
for l in errs: print(f"\n!!!!! errors in {l}:\n" + open(l).read()[-2500:])
assert not errs, "some runs crashed, see above"


In [ ]:
# Tables for the report (also saved in results/revision/*.csv)
!python scripts/revision_tables.py --runs runs --out results/revision


In [ ]:
# Significance: ours (TSR+PTM+ft) vs each baseline, paired over the 15 (court, seed) pairs; needs PART="baselines" finished
import pandas as pd
r = pd.read_csv("runs/all_results.csv")
d = r[r.run.str.startswith("fl_ditto_lam") & r.run.str.contains("_sp_") & (r.seed == 0)]
lam = d.assign(lam=d.run.str.extract(r"_lam([\d.]+)_", expand=False)).groupby("lam").sel_val_loss.mean().idxmin() if len(d) else None
print("Ditto lambda chosen on validation:", lam)
OURS = ("fedavg+ft", "fl_fedavg_ft1.0_sp_mix-dur-typ-disp_")
BASE = [("local_only", "local_sp_"), ("fedprox", "fl_fedprox_mu0.01_sp_"), ("scaffold", "fl_scaffold_sp_"),
        ("fedadam", "fl_fedadam_slr0.003_sp_"), ("fedavg", "fl_fedavg_sp_lh-dur-typ-disp_")]
if lam: BASE.append(("ditto", f"fl_ditto_lam{lam}_sp_"))
for b, rb in BASE:
    if not (r.run.str.contains(rb, regex=False) & (r.method == b)).any(): print("not run yet:", rb); continue
    print(f"\n### ours vs {b} ({rb})")
    !python scripts/significance.py --all runs/all_results.csv --a "{OURS[0]}" --run_a "{OURS[1]}" --b "$b" --run_b "$rb"
